## 6. N-BEATS Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
    !pip install -q pytorch_forecasting
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 37.9 MB/s eta 0:00:00
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import json

import numpy as np

from src.dataloader import load_cgm_stats, load_tensors, load_participant_ids
from src import metrics
from src import train
from src.models.nn_baseline import NBEATSPredictor, HORIZONS

MODEL_NAME = 'NBEATS'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'nbeats'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'nbeats_stability_summary.json'

In [4]:
stats = load_cgm_stats()
train_loader = load_tensors('train')  # batch_size=256 default (Training Procedure)
val_loader = load_tensors('val')
test_loader = load_tensors('test')

test_participant_ids = load_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print(f'number of train batches: ', len(train_loader), '\nnumber of valuation batches: ', len(val_loader),
      '\nnumber of test batches: ', len(test_loader))
print('number of test participants:', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

sample_cgm, sample_target = next(iter(train_loader))
print('\ninput batch shape (cgm):', tuple(sample_cgm.shape))
print('target batch shape:', tuple(sample_target.shape))

number of train batches:  327 
number of valuation batches:  66 
number of test batches:  66
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

input batch shape (cgm): (256, 24)
target batch shape: (256, 5)


In [5]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []  # one entry per seed, in memory -- also written to disk individually below

for seed in SEEDS:
    print(f"Test results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)
    model = NBEATSPredictor(context_length=sample_cgm.shape[1])

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'nbeats' /f'nbeats_seed{seed}',
        verbose=False,
        progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader)
    y_pred_test, y_true_test = train.predict(model, test_loader)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])

    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'nbeats_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = test_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")

Test results — NBEATS | seed=7

device: cuda (NVIDIA A100-SXM4-40GB)  model: NBEATSPredictor  params: 75,312


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.47 | MAE 7.80 | gRMSE 12.53 | Zone A 96.73% | A+B 99.83%
 30 min | RMSE 18.61 | MAE 12.33 | gRMSE 21.39 | Zone A 90.55% | A+B 99.58%
 60 min | RMSE 27.12 | MAE 18.09 | gRMSE 32.87 | Zone A 81.92% | A+B 99.16%
 90 min | RMSE 32.21 | MAE 21.86 | gRMSE 39.68 | Zone A 75.60% | A+B 98.59%
120 min | RMSE 35.17 | MAE 24.02 | gRMSE 43.88 | Zone A 71.46% | A+B 98.21%

--------------------------------------------------------------------------------
Test results — NBEATS | seed=14

device: cuda (NVIDIA A100-SXM4-40GB)  model: NBEATSPredictor  params: 75,312


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.48 | MAE 7.87 | gRMSE 12.62 | Zone A 97.08% | A+B 99.83%
 30 min | RMSE 18.54 | MAE 12.19 | gRMSE 21.43 | Zone A 90.64% | A+B 99.54%
 60 min | RMSE 27.06 | MAE 17.77 | gRMSE 32.93 | Zone A 82.26% | A+B 99.03%
 90 min | RMSE 32.28 | MAE 21.50 | gRMSE 40.18 | Zone A 75.92% | A+B 98.47%
120 min | RMSE 35.17 | MAE 23.86 | gRMSE 44.08 | Zone A 71.62% | A+B 98.16%

--------------------------------------------------------------------------------
Test results — NBEATS | seed=21

device: cuda (NVIDIA A100-SXM4-40GB)  model: NBEATSPredictor  params: 75,312


...

 15 min | RMSE 11.53 | MAE 7.78 | gRMSE 12.87 | Zone A 97.04% | A+B 99.82%
 30 min | RMSE 18.72 | MAE 12.29 | gRMSE 21.87 | Zone A 90.72% | A+B 99.55%
 60 min | RMSE 27.27 | MAE 17.92 | gRMSE 33.41 | Zone A 81.72% | A+B 99.03%
 90 min | RMSE 32.39 | MAE 21.60 | gRMSE 40.39 | Zone A 76.11% | A+B 98.44%
120 min | RMSE 35.27 | MAE 23.84 | gRMSE 44.30 | Zone A 71.77% | A+B 98.10%

--------------------------------------------------------------------------------
Test results — NBEATS | seed=63

device: cuda (NVIDIA A100-SXM4-40GB)  model: NBEATSPredictor  params: 75,312


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.53 | MAE 7.90 | gRMSE 12.63 | Zone A 96.92% | A+B 99.82%
 30 min | RMSE 18.58 | MAE 12.26 | gRMSE 21.31 | Zone A 90.51% | A+B 99.58%
 60 min | RMSE 27.01 | MAE 17.69 | gRMSE 32.70 | Zone A 82.28% | A+B 99.09%
 90 min | RMSE 32.16 | MAE 21.41 | gRMSE 39.76 | Zone A 76.07% | A+B 98.55%
120 min | RMSE 35.14 | MAE 23.70 | gRMSE 43.81 | Zone A 72.10% | A+B 98.21%

--------------------------------------------------------------------------------
Test results — NBEATS | seed=70

device: cuda (NVIDIA A100-SXM4-40GB)  model: NBEATSPredictor  params: 75,312


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.71 | MAE 8.04 | gRMSE 13.24 | Zone A 96.82% | A+B 99.78%
 30 min | RMSE 18.63 | MAE 12.53 | gRMSE 21.59 | Zone A 90.17% | A+B 99.51%
 60 min | RMSE 27.11 | MAE 18.32 | gRMSE 32.79 | Zone A 81.18% | A+B 98.94%
 90 min | RMSE 32.21 | MAE 21.87 | gRMSE 39.80 | Zone A 75.11% | A+B 98.47%
120 min | RMSE 35.13 | MAE 23.91 | gRMSE 43.91 | Zone A 71.54% | A+B 98.13%

--------------------------------------------------------------------------------


In [6]:
def _extract(participant_result, metric):
    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

summary = {}

for h in HORIZONS:
    summary[h] = {}
    for m in METRICS:
        if m == 'zone_a':
            seed_values = [
                r['test_results'][h]['ceg']['A']
            for r in all_results
            ]
        elif m == 'zone_ab':
            seed_values = [
                r['test_results'][h]['ceg']['A']
                + r['test_results'][h]['ceg']['B']
                for r in all_results
            ]
        else:
            seed_values = [
                r['test_results'][h][m]
                for r in all_results
            ]

        mean = float(np.mean(seed_values))
        sd1 = float(np.std(seed_values, ddof=1))

        participant_means = []
        for pid in PARTICIPANT_IDS:
            values_across_seeds = [
                _extract(r['test_results_per_participant'][pid][h], m)
                for r in all_results
            ]
            participant_means.append(np.mean(values_across_seeds))
        sd2 = float(np.std(participant_means, ddof=1))
        summary[h][m] = {
            'mean': mean,
            'sd1_across_seeds': sd1,
            'sd2_across_participants': sd2,
        }

print(
    f'=== {MODEL_NAME} test set: '
    f'overall mean +/- SD_seed (SD_participant) ==='
)

for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min:  "
        f"RMSE = {s['rmse']['mean']:.2f} +/- "
        f"{s['rmse']['sd1_across_seeds']:.2f} "
        f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
        f"MAE = {s['mae']['mean']:.2f} +/- "
        f"{s['mae']['sd1_across_seeds']:.2f} "
        f"({s['mae']['sd2_across_participants']:.2f})   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- "
        f"{s['grmse']['sd1_across_seeds']:.2f} "
        f"({s['grmse']['sd2_across_participants']:.2f})   "
        f"Zone A = {s['zone_a']['mean']:.2f} +/- "
        f"{s['zone_a']['sd1_across_seeds']:.2f} "
        f"({s['zone_a']['sd2_across_participants']:.2f})%   "
        f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
        f"{s['zone_ab']['sd1_across_seeds']:.2f} "
        f"({s['zone_ab']['sd2_across_participants']:.2f})%"
    )


=== NBEATS test set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 11.52 +/- 0.08 (1.89) mg/dL   MAE = 7.88 +/- 0.09 (1.44)   gRMSE = 12.69 +/- 0.27 (2.96)   Zone A = 96.82 +/- 0.15 (2.15)%   Zone A+B = 99.79 +/- 0.03 (0.32)%
 30-min:  RMSE = 18.61 +/- 0.07 (3.70) mg/dL   MAE = 12.39 +/- 0.13 (2.44)   gRMSE = 21.43 +/- 0.25 (6.08)   Zone A = 90.33 +/- 0.26 (3.25)%   Zone A+B = 99.53 +/- 0.03 (0.60)%
 60-min:  RMSE = 27.13 +/- 0.09 (8.48) mg/dL   MAE = 18.05 +/- 0.22 (5.07)   gRMSE = 32.89 +/- 0.29 (13.34)   Zone A = 81.70 +/- 0.43 (4.74)%   Zone A+B = 99.03 +/- 0.06 (0.97)%
 90-min:  RMSE = 32.28 +/- 0.13 (12.09) mg/dL   MAE = 21.74 +/- 0.23 (7.16)   gRMSE = 39.97 +/- 0.34 (18.86)   Zone A = 75.56 +/- 0.54 (5.93)%   Zone A+B = 98.51 +/- 0.08 (1.47)%
120-min:  RMSE = 35.23 +/- 0.17 (14.41) mg/dL   MAE = 23.95 +/- 0.23 (8.77)   gRMSE = 44.05 +/- 0.35 (22.43)   Zone A = 71.50 +/- 0.74 (6.07)%   Zone A+B = 98.17 +/- 0.11 (2.10)%


In [7]:
with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/nbeats_seed*_results.json')
print(f'saved aggregate summary to {STABILITY_SUMMARY_PATH}')


saved per-seed results to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/nbeats/nbeats_seed*_results.json
saved aggregate summary to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/nbeats/nbeats_stability_summary.json
